# Hindi Automatic Speech Recognition
## Speaker-disjoint Conformer–CTC trained from scratch on AI4Bharat IndicVoices

This is the cleaned, final notebook for the project. It replaces the exploratory/debug/recovery notebook with one linear pipeline.

**Base controlled experiment:** 47.465 hours of Hindi training audio, 28,699 training clips, 1,102 training speakers, and a fixed 9.495-hour unseen-speaker test set.

**50-hour result:** 28.45% character error rate (CER) and 64.19% word error rate (WER). This checkpoint was later continued on 100K and 200K clips; the final 200K test result was 22.80% CER and 53.71% WER.

## Experimental design

- **Dataset:** AI4Bharat IndicVoices, Hindi speech with human transcripts.
- **Audio quality controls:** mono 16 kHz FLAC; clips outside 0.5–30 seconds and `<unintelligible>` transcripts were removed.
- **Evaluation integrity:** speakers are disjoint across training, validation, and test splits.
- **Nested experiments:** the 10h set is contained in the 25h set, which is contained in the 50h set. This isolates the effect of adding training data.
- **Input representation:** normalized 80-bin log-Mel spectrograms.
- **Model:** two-layer convolutional subsampling, eight Conformer blocks, and a 67-class character CTC head (66 characters plus blank), totaling approximately 4.5 million trainable parameters.
- **Decoding:** greedy CTC decoding without an external language model.

## How to use this notebook

1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Run the cells from top to bottom.
3. Set `EXPERIMENT = "50h"` for the final model. The same notebook can also restore the 10h and 25h experiments.
4. Training and evaluation are protected by switches, so opening the notebook cannot accidentally start a long job.

The notebook assumes the persistent project artifacts already exist in:

```text
/content/drive/MyDrive/Hindi_ASR_Project
```

The original exploratory notebook is still useful as a record of raw dataset discovery and one-time Parquet extraction. Those duplicate and failed cells are intentionally not repeated here.

## 1. Environment and experiment configuration

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import json
import math
import os
import random
import shutil
import tarfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from torchaudio.models import Conformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU before continuing.")

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Change only this value to restore another completed experiment.
EXPERIMENT = "50h"  # "10h", "25h", or "50h"

PROJECT_DIR = Path("/content/drive/MyDrive/Hindi_ASR_Project")

EXPERIMENTS = {
    "10h": {
        "expected_train": 5722,
        "expected_total_audio": 6289,
        "train_csv": "train_10h_ctc_manifest.csv",
        "parts_dir": "hindi_asr_10h_parts",
        "local_root": "/content/hindi_asr_10h",
        "checkpoint_dir": "checkpoints_10h",
        "checkpoint_prefix": "conformer_10h",
    },
    "25h": {
        "expected_train": 14352,
        "expected_total_audio": 14919,
        "train_csv": "train_25h_ctc_manifest.csv",
        "parts_dir": "hindi_asr_25h_parts",
        "local_root": "/content/hindi_asr_25h",
        "checkpoint_dir": "checkpoints_25h",
        "checkpoint_prefix": "conformer_25h",
    },
    "50h": {
        "expected_train": 28699,
        "expected_total_audio": 29266,
        "train_csv": "train_50h_ctc_manifest.csv",
        "parts_dir": "hindi_asr_50h_parts",
        "local_root": "/content/hindi_asr_50h",
        "checkpoint_dir": "checkpoints_50h",
        "checkpoint_prefix": "conformer_50h",
    },
}

cfg = EXPERIMENTS[EXPERIMENT]
PARTS_DIR = PROJECT_DIR / cfg["parts_dir"]
LOCAL_ROOT = Path(cfg["local_root"])
LOCAL_AUDIO = LOCAL_ROOT / "audio"
TRAIN_JSON = LOCAL_ROOT / "train_manifest.json"
VALIDATION_JSON = LOCAL_ROOT / "validation_manifest.json"
TRAIN_CSV = PROJECT_DIR / cfg["train_csv"]
VALIDATION_CSV = PROJECT_DIR / "validation_1h_ctc_manifest.csv"
VOCAB_PATH = PROJECT_DIR / "ctc_vocabulary.json"

CHECKPOINT_DIR = PROJECT_DIR / cfg["checkpoint_dir"]
LAST_CHECKPOINT = CHECKPOINT_DIR / f'{cfg["checkpoint_prefix"]}_last.pt'
BEST_CHECKPOINT = CHECKPOINT_DIR / f'{cfg["checkpoint_prefix"]}_best.pt'
SELECTED_CHECKPOINT = CHECKPOINT_DIR / f'{cfg["checkpoint_prefix"]}_selected_epoch30.pt'
HISTORY_PATH = CHECKPOINT_DIR / "training_history.csv"

print("Experiment:", EXPERIMENT)
print("Project folder:", PROJECT_DIR)
print("Expected training recordings:", cfg["expected_train"])

## 2. Restore prepared audio and rebuild local manifests

Google Colab deletes `/content` when the runtime resets. The audio archives and metadata remain safe in Drive. This cell restores them and recreates the local JSON manifests automatically.

In [ ]:
LOCAL_AUDIO.mkdir(parents=True, exist_ok=True)

audio_files = sorted(LOCAL_AUDIO.glob("*.flac"))
expected_audio = cfg["expected_total_audio"]

if len(audio_files) != expected_audio:
    archive_parts = sorted(PARTS_DIR.glob("audio_part_*.tar"))
    if not archive_parts:
        archive_parts = sorted(PARTS_DIR.glob("*.tar"))

    if len(archive_parts) != 15:
        raise FileNotFoundError(
            f"Expected 15 archive parts in {PARTS_DIR}, found {len(archive_parts)}."
        )

    print(f"Restoring {len(archive_parts)} archive parts...")
    for index, archive_path in enumerate(archive_parts, start=1):
        with tarfile.open(archive_path, "r") as archive:
            archive.extractall("/content", filter="data")
        print(f"Restored {index}/{len(archive_parts)}")

audio_files = sorted(LOCAL_AUDIO.glob("*.flac"))
if len(audio_files) != expected_audio:
    raise ValueError(f"Expected {expected_audio} audio files, found {len(audio_files)}.")

def write_local_manifest(csv_path, json_path, audio_dir=LOCAL_AUDIO):
    metadata = pd.read_csv(csv_path)
    records = []

    for row in metadata.itertuples(index=False):
        original_row = int(row.original_row)
        audio_path = Path(audio_dir) / f"clip_{original_row:07d}.flac"

        if not audio_path.exists():
            raise FileNotFoundError(f"Missing audio: {audio_path}")

        records.append({
            "audio_filepath": str(audio_path),
            "text": str(row.model_text),
            "duration": float(row.duration),
            "speaker_id": str(row.speaker_id),
            "original_row": original_row,
        })

    with open(json_path, "w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

    return records

train_records = write_local_manifest(TRAIN_CSV, TRAIN_JSON)
validation_records = write_local_manifest(VALIDATION_CSV, VALIDATION_JSON)

if len(train_records) != cfg["expected_train"]:
    raise ValueError(
        f'Expected {cfg["expected_train"]} training records, found {len(train_records)}.'
    )
if len(validation_records) != 567:
    raise ValueError(f"Expected 567 validation records, found {len(validation_records)}.")

print("Audio files ready:", len(audio_files))
print("Training records:", len(train_records))
print("Validation records:", len(validation_records))

## 3. Verify speaker-disjoint splits

In [ ]:
train_metadata = pd.read_csv(TRAIN_CSV)
validation_metadata = pd.read_csv(VALIDATION_CSV)
test_metadata = pd.read_csv(PROJECT_DIR / "test_10h_ctc_manifest.csv")

train_speakers = set(train_metadata["speaker_id"].astype(str))
validation_speakers = set(validation_metadata["speaker_id"].astype(str))
test_speakers = set(test_metadata["speaker_id"].astype(str))

split_summary = pd.DataFrame([
    {
        "split": "train",
        "clips": len(train_metadata),
        "speakers": len(train_speakers),
        "hours": train_metadata["duration"].sum() / 3600,
    },
    {
        "split": "validation",
        "clips": len(validation_metadata),
        "speakers": len(validation_speakers),
        "hours": validation_metadata["duration"].sum() / 3600,
    },
    {
        "split": "test",
        "clips": len(test_metadata),
        "speakers": len(test_speakers),
        "hours": test_metadata["duration"].sum() / 3600,
    },
])

display(split_summary.round({"hours": 3}))

overlaps = {
    "train-validation": len(train_speakers & validation_speakers),
    "train-test": len(train_speakers & test_speakers),
    "validation-test": len(validation_speakers & test_speakers),
}
print("Speaker overlaps:", overlaps)
assert all(value == 0 for value in overlaps.values())

## 4. Vocabulary, features, dataset, and Conformer–CTC model

In [ ]:
with open(VOCAB_PATH, "r", encoding="utf-8") as file:
    vocabulary_data = json.load(file)

char_to_id = None
for key in ["char_to_id", "character_to_id", "token_to_id", "vocabulary", "vocab"]:
    value = vocabulary_data.get(key)
    if isinstance(value, dict):
        candidate = {
            character: int(token_id)
            for character, token_id in value.items()
            if isinstance(character, str) and len(character) == 1
        }
        if candidate:
            char_to_id = candidate
            break

if char_to_id is None:
    char_to_id = {
        character: index + 1
        for index, character in enumerate(vocabulary_data["characters"])
    }

blank_id = int(vocabulary_data.get("blank_id", 0))
id_to_char = {token_id: character for character, token_id in char_to_id.items()}
number_of_classes = max(max(char_to_id.values()), blank_id) + 1

print("Character tokens:", len(char_to_id))
print("CTC blank ID:", blank_id)
print("Output classes:", number_of_classes)

In [ ]:
class HindiASRDataset(Dataset):
    def __init__(self, manifest_path):
        with open(manifest_path, "r", encoding="utf-8") as file:
            self.records = [json.loads(line) for line in file if line.strip()]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        waveform, sample_rate = sf.read(record["audio_filepath"], dtype="float32")

        if sample_rate != 16000:
            raise ValueError(f"Expected 16 kHz audio, received {sample_rate} Hz.")

        waveform = torch.from_numpy(waveform)
        if waveform.ndim == 2:
            waveform = waveform.mean(dim=1)

        text = record["text"]
        target = torch.tensor(
            [char_to_id[character] for character in text], dtype=torch.long
        )
        return waveform, target, text


mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=400,
    win_length=400,
    hop_length=160,
    n_mels=80,
    power=2.0,
)


def collate_batch(batch):
    waveforms, targets, texts = zip(*batch)
    waveform_lengths = torch.tensor([len(waveform) for waveform in waveforms])
    padded_waveforms = pad_sequence(waveforms, batch_first=True, padding_value=0.0)

    features = mel_transform(padded_waveforms)
    features = torch.log(features.clamp_min(1e-10)).transpose(1, 2)
    feature_lengths = waveform_lengths // 160 + 1

    for index, length in enumerate(feature_lengths):
        valid = features[index, :int(length)]
        features[index, :int(length)] = (
            valid - valid.mean(dim=0, keepdim=True)
        ) / valid.std(dim=0, keepdim=True).clamp_min(1e-5)

    return {
        "features": features,
        "feature_lengths": feature_lengths,
        "targets": pad_sequence(targets, batch_first=True, padding_value=blank_id),
        "target_lengths": torch.tensor([len(target) for target in targets]),
        "texts": texts,
    }

In [ ]:
class ConvSubsampling(nn.Module):
    def __init__(self, input_features=80, model_dimension=144):
        super().__init__()
        self.convolutions = nn.Sequential(
            nn.Conv2d(1, model_dimension, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(
                model_dimension, model_dimension, kernel_size=3, stride=2, padding=1
            ),
            nn.ReLU(),
        )

        reduced_frequency = (input_features + 1) // 2
        reduced_frequency = (reduced_frequency + 1) // 2
        self.projection = nn.Linear(
            model_dimension * reduced_frequency, model_dimension
        )
        self.dropout = nn.Dropout(0.1)

    def forward(self, features, lengths):
        features = self.convolutions(features.unsqueeze(1))
        batch, channels, time_steps, frequency = features.shape
        features = features.permute(0, 2, 1, 3).contiguous()
        features = features.view(batch, time_steps, channels * frequency)
        features = self.dropout(self.projection(features))
        lengths = (lengths + 1) // 2
        lengths = (lengths + 1) // 2
        return features, lengths


class HindiConformerCTC(nn.Module):
    def __init__(self, output_classes):
        super().__init__()
        self.subsampling = ConvSubsampling()
        self.conformer = Conformer(
            input_dim=144,
            num_heads=4,
            ffn_dim=576,
            num_layers=8,
            depthwise_conv_kernel_size=31,
            dropout=0.1,
            use_group_norm=True,
            convolution_first=False,
        )
        self.output_layer = nn.Linear(144, output_classes)

    def forward(self, features, lengths):
        encoded, lengths = self.subsampling(features, lengths)
        encoded, lengths = self.conformer(encoded, lengths)
        return self.output_layer(encoded), lengths


model = HindiConformerCTC(number_of_classes).to(device)
ctc_loss_function = nn.CTCLoss(
    blank=blank_id, reduction="mean", zero_infinity=True
)

parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", round(parameter_count / 1_000_000, 2), "million")

## 5. Data loaders and smoke test

In [ ]:
train_dataset = HindiASRDataset(TRAIN_JSON)
validation_dataset = HindiASRDataset(VALIDATION_JSON)

loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    generator=loader_generator,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_batch,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_batch,
)

test_batch = next(iter(train_loader))
with torch.inference_mode():
    test_logits, test_lengths = model(
        test_batch["features"].to(device),
        test_batch["feature_lengths"].to(device),
    )

print("Training recordings:", len(train_dataset))
print("Validation recordings:", len(validation_dataset))
print("Feature shape:", tuple(test_batch["features"].shape))
print("Model output shape:", tuple(test_logits.shape))

## 6. Resumable training

The cell below defines the training system but does not start training. Checkpoints are written after every completed epoch, so a disconnected runtime loses at most the current unfinished epoch.

Set `RUN_TRAINING = True` only when you intentionally want to train or resume.

In [ ]:
TOTAL_PLANNED_EPOCHS = 30
GRADIENT_ACCUMULATION_STEPS = 8
BASE_LEARNING_RATE = 3e-4


def apply_spec_augment(features, lengths):
    augmented = features.clone()
    batch_size, _, feature_count = augmented.shape

    for index in range(batch_size):
        valid_length = int(lengths[index])

        for _ in range(2):
            width = random.randint(0, 8)
            if width > 0:
                start = random.randint(0, feature_count - width)
                augmented[index, :valid_length, start:start + width] = 0

        maximum_time_width = min(40, max(1, int(valid_length * 0.05)))
        for _ in range(2):
            width = random.randint(0, maximum_time_width)
            if width > 0 and valid_length > width:
                start = random.randint(0, valid_length - width)
                augmented[index, start:start + width, :] = 0

    return augmented


def edit_distance(reference, prediction):
    previous = list(range(len(prediction) + 1))
    for reference_index, reference_item in enumerate(reference, start=1):
        current = [reference_index]
        for prediction_index, prediction_item in enumerate(prediction, start=1):
            current.append(min(
                current[prediction_index - 1] + 1,
                previous[prediction_index] + 1,
                previous[prediction_index - 1]
                + (reference_item != prediction_item),
            ))
        previous = current
    return previous[-1]


def greedy_decode(logits, output_lengths):
    sequences = logits.argmax(dim=-1)
    predictions = []

    for sequence, length in zip(sequences, output_lengths):
        characters = []
        previous = None
        for token in sequence[:int(length)].tolist():
            if token != blank_id and token != previous and token in id_to_char:
                characters.append(id_to_char[token])
            previous = token
        predictions.append("".join(characters).strip())

    return predictions


def calculate_error_rates(references, predictions):
    character_errors = reference_characters = 0
    word_errors = reference_words = 0

    for reference, prediction in zip(references, predictions):
        character_errors += edit_distance(list(reference), list(prediction))
        reference_characters += len(reference)
        reference_word_list = reference.split()
        prediction_word_list = prediction.split()
        word_errors += edit_distance(reference_word_list, prediction_word_list)
        reference_words += len(reference_word_list)

    return (
        character_errors / max(1, reference_characters),
        word_errors / max(1, reference_words),
    )


def evaluate_loader(loader):
    model.eval()
    total_loss = 0.0
    references, predictions = [], []

    with torch.inference_mode():
        for batch in loader:
            features = batch["features"].to(device, non_blocking=True)
            feature_lengths = batch["feature_lengths"].to(device)
            targets = batch["targets"].to(device, non_blocking=True)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits, output_lengths = model(features, feature_lengths)

            loss = ctc_loss_function(
                logits.float().log_softmax(dim=-1).transpose(0, 1),
                targets,
                output_lengths.cpu(),
                batch["target_lengths"],
            )
            total_loss += float(loss.cpu())
            predictions.extend(greedy_decode(logits.cpu(), output_lengths.cpu()))
            references.extend(batch["texts"])

    cer, wer = calculate_error_rates(references, predictions)
    return {
        "loss": total_loss / len(loader),
        "cer": cer,
        "wer": wer,
        "reference": references[0],
        "prediction": predictions[0],
    }

In [ ]:
def save_safely(data, destination):
    destination = Path(destination)
    local_file = Path("/content/temporary_asr_checkpoint.pt")
    temporary_drive_file = Path(str(destination) + ".temporary")
    torch.save(data, local_file)
    shutil.copy2(local_file, temporary_drive_file)
    os.replace(temporary_drive_file, destination)
    local_file.unlink(missing_ok=True)


def train_until(target_epoch=30):
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=BASE_LEARNING_RATE, weight_decay=1e-3
    )
    updates_per_epoch = math.ceil(
        len(train_loader) / GRADIENT_ACCUMULATION_STEPS
    )
    warmup_steps = round(2.8 * updates_per_epoch)
    total_update_steps = TOTAL_PLANNED_EPOCHS * updates_per_epoch

    def learning_rate_multiplier(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(
            1, total_update_steps - warmup_steps
        )
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return 0.1 + 0.9 * cosine

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=learning_rate_multiplier
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    start_epoch = 1
    global_step = 0
    best_validation_loss = float("inf")
    history = []

    if LAST_CHECKPOINT.exists():
        checkpoint = torch.load(
            LAST_CHECKPOINT, map_location=device, weights_only=False
        )
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        global_step = checkpoint["global_step"]
        best_validation_loss = checkpoint["best_validation_loss"]
        history = checkpoint.get("history", [])
        random.seed(SEED + start_epoch)
        loader_generator.manual_seed(SEED + start_epoch)
        print("Resuming after epoch", checkpoint["epoch"])
    else:
        print("Starting a new", EXPERIMENT, "training run")

    for epoch in range(start_epoch, target_epoch + 1):
        epoch_start = time.time()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_training_loss = 0.0

        for batch_number, batch in enumerate(train_loader, start=1):
            features = batch["features"].to(device, non_blocking=True)
            feature_lengths = batch["feature_lengths"].to(device)
            targets = batch["targets"].to(device, non_blocking=True)

            if epoch >= 2:
                features = apply_spec_augment(features, feature_lengths)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits, output_lengths = model(features, feature_lengths)
                loss = ctc_loss_function(
                    logits.float().log_softmax(dim=-1).transpose(0, 1),
                    targets,
                    output_lengths.cpu(),
                    batch["target_lengths"],
                )
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

            if not torch.isfinite(loss):
                raise ValueError(
                    f"Non-finite loss at epoch {epoch}, batch {batch_number}"
                )

            scaler.scale(scaled_loss).backward()
            should_update = (
                batch_number % GRADIENT_ACCUMULATION_STEPS == 0
                or batch_number == len(train_loader)
            )

            if should_update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1

            total_training_loss += float(loss.detach().cpu())
            if batch_number % 1000 == 0:
                print(
                    f"Epoch {epoch} | batch {batch_number}/{len(train_loader)} "
                    f"| loss {float(loss.detach().cpu()):.3f}"
                )

        training_loss = total_training_loss / len(train_loader)
        validation_result = evaluate_loader(validation_loader)
        epoch_minutes = (time.time() - epoch_start) / 60
        learning_rate = scheduler.get_last_lr()[0]

        history.append({
            "epoch": epoch,
            "training_loss": training_loss,
            "validation_loss": validation_result["loss"],
            "validation_cer": validation_result["cer"],
            "validation_wer": validation_result["wer"],
            "learning_rate": learning_rate,
            "minutes": epoch_minutes,
        })

        improved = validation_result["loss"] < best_validation_loss
        if improved:
            best_validation_loss = validation_result["loss"]
            save_safely({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "validation_loss": best_validation_loss,
                "character_to_id": char_to_id,
                "blank_id": blank_id,
            }, BEST_CHECKPOINT)

        save_safely({
            "epoch": epoch,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_validation_loss": best_validation_loss,
            "history": history,
        }, LAST_CHECKPOINT)

        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
        print(
            f"Epoch {epoch}/{TOTAL_PLANNED_EPOCHS} | "
            f"train loss {training_loss:.4f} | "
            f"val loss {validation_result['loss']:.4f} | "
            f"CER {validation_result['cer'] * 100:.2f}% | "
            f"WER {validation_result['wer'] * 100:.2f}% | "
            f"{epoch_minutes:.2f} min"
        )

    print("Training stage finished. Last checkpoint:", LAST_CHECKPOINT)

In [ ]:
RUN_TRAINING = False
TARGET_EPOCH = 30

if RUN_TRAINING:
    train_until(TARGET_EPOCH)
else:
    print("Training skipped. Set RUN_TRAINING = True only when needed.")

## 7. Training history and checkpoint selection

In [ ]:
if not HISTORY_PATH.exists():
    raise FileNotFoundError(f"Training history not found: {HISTORY_PATH}")

history = pd.read_csv(HISTORY_PATH)
latest = history.iloc[-1]
best_loss = history.loc[history["validation_loss"].idxmin()]
best_cer = history.loc[history["validation_cer"].idxmin()]
best_wer = history.loc[history["validation_wer"].idxmin()]

summary = pd.DataFrame([
    {
        "selection": "latest",
        "epoch": int(latest["epoch"]),
        "loss": latest["validation_loss"],
        "CER (%)": latest["validation_cer"] * 100,
        "WER (%)": latest["validation_wer"] * 100,
    },
    {
        "selection": "best validation loss",
        "epoch": int(best_loss["epoch"]),
        "loss": best_loss["validation_loss"],
        "CER (%)": best_loss["validation_cer"] * 100,
        "WER (%)": best_loss["validation_wer"] * 100,
    },
    {
        "selection": "best CER",
        "epoch": int(best_cer["epoch"]),
        "loss": best_cer["validation_loss"],
        "CER (%)": best_cer["validation_cer"] * 100,
        "WER (%)": best_cer["validation_wer"] * 100,
    },
    {
        "selection": "best WER",
        "epoch": int(best_wer["epoch"]),
        "loss": best_wer["validation_loss"],
        "CER (%)": best_wer["validation_cer"] * 100,
        "WER (%)": best_wer["validation_wer"] * 100,
    },
])
display(summary.round(3))

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["training_loss"], label="training")
axes[0].plot(history["epoch"], history["validation_loss"], label="validation")
axes[0].set(title=f"{EXPERIMENT} loss", xlabel="Epoch", ylabel="CTC loss")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history["epoch"], history["validation_cer"] * 100, label="CER")
axes[1].plot(history["epoch"], history["validation_wer"] * 100, label="WER")
axes[1].set(title=f"{EXPERIMENT} validation errors", xlabel="Epoch", ylabel="Error (%)")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Restore the fixed unseen-speaker test set

The same 9.495-hour test set is used for all three experiments. This makes the 10h, 25h, and 50h comparison fair.

In [ ]:
TEST_PARTS_DIR = PROJECT_DIR / "hindi_asr_test_10h_parts"
TEST_LOCAL_ROOT = Path("/content/hindi_asr_test_10h")
TEST_AUDIO_DIR = TEST_LOCAL_ROOT / "audio"
TEST_JSON = TEST_LOCAL_ROOT / "test_manifest.json"
TEST_CSV = PROJECT_DIR / "test_10h_ctc_manifest.csv"
TEST_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

test_audio_files = sorted(TEST_AUDIO_DIR.glob("*.flac"))
if len(test_audio_files) != 5897:
    test_parts = sorted(TEST_PARTS_DIR.glob("*.tar"))
    if len(test_parts) != 15:
        raise FileNotFoundError(
            f"Expected 15 test archive parts, found {len(test_parts)}."
        )
    for index, archive_path in enumerate(test_parts, start=1):
        with tarfile.open(archive_path, "r") as archive:
            archive.extractall("/content", filter="data")
        print(f"Restored test part {index}/{len(test_parts)}")

test_audio_files = sorted(TEST_AUDIO_DIR.glob("*.flac"))
if len(test_audio_files) != 5897:
    raise ValueError(f"Expected 5897 test files, found {len(test_audio_files)}.")

test_records = write_local_manifest(TEST_CSV, TEST_JSON, TEST_AUDIO_DIR)
if len(test_records) != 5897:
    raise ValueError(f"Expected 5897 test records, found {len(test_records)}.")

print("Test recordings ready:", len(test_records))

## 9. Final test evaluation

In [ ]:
def evaluate_test(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.to(device).eval()

    test_dataset = HindiASRDataset(TEST_JSON)
    test_loader = DataLoader(
        test_dataset,
        batch_size=4,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_batch,
    )

    total_loss = 0.0
    total_recordings = 0
    character_errors = reference_characters = 0
    word_errors = reference_words = 0
    rows = []
    started = time.time()

    with torch.inference_mode():
        for batch_number, batch in enumerate(test_loader, start=1):
            features = batch["features"].to(device, non_blocking=True)
            feature_lengths = batch["feature_lengths"].to(device)
            targets = batch["targets"].to(device, non_blocking=True)
            target_lengths = batch["target_lengths"].to(device)

            logits, output_lengths = model(features, feature_lengths)
            log_probabilities = F.log_softmax(logits, dim=-1)
            loss = ctc_loss_function(
                log_probabilities.transpose(0, 1),
                targets,
                output_lengths,
                target_lengths,
            )

            batch_size = features.size(0)
            total_loss += float(loss) * batch_size
            total_recordings += batch_size
            predictions = greedy_decode(logits, output_lengths)

            for reference, prediction in zip(batch["texts"], predictions):
                char_edits = edit_distance(list(reference), list(prediction))
                word_edits = edit_distance(reference.split(), prediction.split())
                character_errors += char_edits
                reference_characters += len(reference)
                word_errors += word_edits
                reference_words += len(reference.split())
                rows.append({
                    "reference": reference,
                    "prediction": prediction,
                    "character_errors": char_edits,
                    "reference_characters": len(reference),
                    "word_errors": word_edits,
                    "reference_words": len(reference.split()),
                })

            if batch_number % 250 == 0:
                print(f"Evaluated {batch_number}/{len(test_loader)} batches")

    elapsed = time.time() - started
    test_hours = sum(record["duration"] for record in test_records) / 3600
    results = {
        "checkpoint_epoch": int(checkpoint["epoch"]),
        "test_recordings": total_recordings,
        "test_audio_hours": test_hours,
        "test_loss": total_loss / total_recordings,
        "test_cer": character_errors / reference_characters,
        "test_wer": word_errors / reference_words,
        "evaluation_minutes": elapsed / 60,
        "real_time_factor": elapsed / (test_hours * 3600),
        "checkpoint_path": str(checkpoint_path),
    }

    result_path = PROJECT_DIR / f"test_{EXPERIMENT}_results.json"
    prediction_path = PROJECT_DIR / f"test_{EXPERIMENT}_predictions.csv"
    with open(result_path, "w", encoding="utf-8") as file:
        json.dump(results, file, ensure_ascii=False, indent=2)
    pd.DataFrame(rows).to_csv(prediction_path, index=False, encoding="utf-8")

    print(json.dumps(results, indent=2))
    print("\nExamples:")
    for row in rows[:5]:
        print("Reference :", row["reference"])
        print("Prediction:", row["prediction"], "\n")
    return results, pd.DataFrame(rows)

In [ ]:
RUN_EVALUATION = False

# The 50h project selected epoch 30 because it had the lowest validation
# CER and WER. For other experiments, use the chosen checkpoint in Drive.
evaluation_checkpoint = (
    SELECTED_CHECKPOINT if SELECTED_CHECKPOINT.exists() else LAST_CHECKPOINT
)

print("Evaluation checkpoint:", evaluation_checkpoint)
if RUN_EVALUATION:
    final_results, test_predictions = evaluate_test(evaluation_checkpoint)
else:
    print("Evaluation skipped. Set RUN_EVALUATION = True to run it.")

## 10. Compare the 10h, 25h, and 50h experiments

In [ ]:
comparison_rows = []
for label in ["10h", "25h", "50h"]:
    result_path = PROJECT_DIR / f"test_{label}_results.json"
    if result_path.exists():
        with open(result_path, "r", encoding="utf-8") as file:
            result = json.load(file)
        comparison_rows.append({
            "training_data": label,
            "CER (%)": result["test_cer"] * 100,
            "WER (%)": result["test_wer"] * 100,
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(2))

if len(comparison) == 3:
    axis = comparison.set_index("training_data")[["CER (%)", "WER (%)"]].plot(
        kind="bar", figsize=(9, 5), color=["#2878B5", "#E07A5F"]
    )
    axis.set_title("Hindi ASR accuracy improves with more training data")
    axis.set_xlabel("Training data")
    axis.set_ylabel("Error rate (%) — lower is better")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(frameon=False)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 11. Transcribe one new WAV or FLAC recording

Set `RUN_DEMO = True`, run the cell, and upload a Hindi recording. Audio is converted to mono 16 kHz before inference. Greedy CTC decoding is used, so punctuation is not produced.

In [ ]:
RUN_DEMO = False

if RUN_DEMO:
    from google.colab import files

    uploaded = files.upload()
    uploaded_path = next(iter(uploaded))
    waveform, sample_rate = sf.read(uploaded_path, dtype="float32")
    waveform = torch.from_numpy(waveform)

    if waveform.ndim == 2:
        waveform = waveform.mean(dim=1)
    if sample_rate != 16000:
        waveform = torchaudio.functional.resample(waveform, sample_rate, 16000)

    features = mel_transform(waveform.unsqueeze(0))
    features = torch.log(features.clamp_min(1e-10)).transpose(1, 2)
    feature_length = torch.tensor([features.shape[1]])
    valid = features[0, :feature_length.item()]
    features[0, :feature_length.item()] = (
        valid - valid.mean(dim=0, keepdim=True)
    ) / valid.std(dim=0, keepdim=True).clamp_min(1e-5)

    checkpoint = torch.load(
        evaluation_checkpoint, map_location=device, weights_only=False
    )
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()

    with torch.inference_mode():
        logits, output_lengths = model(features.to(device), feature_length.to(device))
    transcription = greedy_decode(logits, output_lengths)[0]
    print("Hindi transcription:", transcription)
else:
    print("Demo skipped. Set RUN_DEMO = True when you want to upload audio.")

## Base experiments and final continuation result

| Training data | Test CER | Test WER |
|---|---:|---:|
| 10 hours | 46.67% | 91.37% |
| 25 hours | 36.33% | 78.35% |
| 50 hours | 28.45% | 64.19% |
| 200K clips (continued training) | **22.80%** | **53.71%** |

The 50-hour checkpoint was the end of the controlled notebook experiment. It was then warm-started on 100K clips for 12 continuation epochs, followed by three completed 200K continuation epochs. All reported test results use the same speaker-disjoint test set. The remaining errors are concentrated in long utterances, uncommon names and numbers, similar Hindi phonemes, and word-boundary decisions.

### Saved base-stage artifacts

- Selected model: `checkpoints_50h/conformer_50h_selected_epoch30.pt`
- Test metrics: `test_50h_results.json`
- Per-utterance predictions: `test_50h_predictions.csv`
- Training history: `checkpoints_50h/training_history.csv`